# 2b. Predicting deception with a probe (with pre-computed S3 activations)

This is the **alternate-data companion** to [`2-predicting.ipynb`](2-predicting.ipynb). The main notebook extracts activations itself by running a model on NDIF; this one skips that step and loads **pre-computed residual-stream activations from a public S3 bucket** — so it needs **no GPU and no API keys**.

Everything downstream is identical: train a per-layer linear probe (§2), then evaluate how well it generalizes to a *different* scenario (§3). Only the data source in §1 differs. For the raw prompt + response pairs and the extract-it-yourself workflow, see the main notebook.

## 0️⃣ Setup

A small stack: `zarr`/`s3fs`/`boto3` to read the S3 activations, plus the probing libraries. No `nnsight`, no credentials.

In [ ]:
# Data + probing stack — no nnsight needed (we don't run a model here), no keys needed (the bucket is public).
!pip install -q "zarr>=3.0" "s3fs>=2024.0" boto3 pandas pyarrow numpy scikit-learn matplotlib

from IPython.display import clear_output
clear_output()
print("Setup complete.")

## 1️⃣ Load pre-computed activations from S3

**Liars' Bench Expanded** lives in a public S3 bucket and pairs every prompt + response with the model's **residual-stream activations**, captured at inference time via NDIF. The bucket is organized by `captured_model` -> `scenario` -> `split`, with metadata and tensors side by side:

```
s3://liars-bench-expanded/
  metadata/captured_model=<model>/scenario=<scenario>/split=<split>/records.parquet   # labels + prompt + response pairs
  data/    captured_model=<model>/scenario=<scenario>/split=<split>/activations.zarr/  # activation tensors
```

Two file formats, each chosen for a reason:

- **`records.parquet`** — one row per prompt + response: the `deceptive` label, the full `messages` conversation, the generating `model`, and shape info. Parquet is columnar, so you can read just the `deceptive` column without touching the prompt + response text.
- **`activations.zarr`** — a tensor of shape `(n_records, n_layers, hidden_dim)`, `float16`, row-aligned with the Parquet. Zarr is chunked by `(1000 records, 1 layer, full hidden)`, so reading **one layer across all records** — exactly what probe training needs — downloads only that slice instead of the whole array.

`captured_model` is the natural unit of work: a probe reads one specific model's activations, so you always pick a model first. Let's connect and look around.

In [ ]:
import boto3
import numpy as np
import pandas as pd
import zarr
import s3fs
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "liars-bench-expanded" # TODO: bucket not available yet

# Anonymous clients — the bucket is public, no AWS credentials needed.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
fs = s3fs.S3FileSystem(anon=True)

def list_prefixes(prefix):
    """List the immediate child 'folders' under an S3 prefix."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, Delimiter="/")
    return [p["Prefix"] for p in resp.get("CommonPrefixes", [])]

# Which models' activations are available?
print("Captured models:")
for p in list_prefixes("data/"):
    print("  ", p.split("captured_model=")[-1].rstrip("/"))

In [ ]:
# Pick a captured model, then see which scenarios (generator + source dataset) it has.
MODEL = "google/gemma-3-27b-it"

print(f"Scenarios for captured_model={MODEL}:")
for p in list_prefixes(f"data/captured_model={MODEL}/"):
    print("  ", p.split("scenario=")[-1].rstrip("/"))

Now load one `(model, scenario, split)`: labels from the Parquet, activations from the Zarr. Opening the Zarr is instantaneous — it reads only the header until we actually index into it.

In [ ]:
SCENARIO = "qwen-2.5-72b-instruct_wmdp-bio-2026"
SPLIT    = "test"

base = f"captured_model={MODEL}/scenario={SCENARIO}/split={SPLIT}"

# --- metadata: labels + prompt + response pairs ---
df = pd.read_parquet(f"s3://{BUCKET}/metadata/{base}/records.parquet", filesystem=fs)
labels = df["deceptive"].values
print(f"{len(df)} records | {labels.sum()} deceptive / {(~labels).sum()} honest")
print("columns:", list(df.columns))

# --- activations: opening reads only the header ---
store = zarr.open(f"s3://{BUCKET}/data/{base}/activations.zarr", mode="r", storage_options={"anon": True})
print(f"\nactivations: shape {store.shape}  (n_records, n_layers, hidden_dim), dtype {store.dtype}")

# Fetch a single layer across all records — Zarr downloads only that slice.
LAYER = 15
acts_layer = store[:, LAYER, :].astype(np.float32)   # (n_records, hidden_dim)
print(f"layer {LAYER}: {acts_layer.shape}  ({acts_layer.nbytes/1e6:.1f} MB)")

## 2️⃣ Training a probe to predict deception

A **linear probe** is just a logistic regression trained on a layer's activation to predict the `deceptive` label. If it works, that layer's residual stream **linearly encodes** whether the model is lying.

Why linear? It's deliberately weak. A powerful classifier could *memorize* surface quirks of these particular prompt + response pairs; a linear probe can only succeed if the deception signal is genuinely present in the representation. That makes it both a strong baseline and an honest measurement.

### 2.1 Look before you train

Before fitting anything, look at the data. If honest and deceptive responses occupy different regions of activation space, a probe will find them — and **PCA** is the quickest way to see whether that structure is there. PCA projects each high-dimensional activation down to 2D along the directions of greatest variance; if the two classes separate even in those two dimensions, the signal is strong and linearly accessible.

We'll (1) check the label balance, (2) project one layer to 2D colored by label, and (3) watch separation emerge across depth.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

n_layers = store.shape[1]
DECEPTIVE_C, HONEST_C = "#e74c3c", "#2ecc71"

# (1) Label balance — a skewed split inflates accuracy, so always check first.
n_dec, n_hon = int(labels.sum()), int((~labels).sum())
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.bar(["Deceptive", "Honest"], [n_dec, n_hon], color=[DECEPTIVE_C, HONEST_C], edgecolor="white")
ax.set_ylabel("records"); ax.set_title(f"Label balance\n{MODEL} - {SCENARIO}", fontsize=10)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
print(f"{n_dec} deceptive / {n_hon} honest")

In [ ]:
# (2) PCA of one layer's activations, colored by label.
acts = store[:, LAYER, :].astype(np.float32)
coords = PCA(n_components=2).fit_transform(acts)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for lab, c, name in [(True, DECEPTIVE_C, "Deceptive"), (False, HONEST_C, "Honest")]:
    m = labels == lab
    ax.scatter(coords[m, 0], coords[m, 1], c=c, label=name, alpha=0.55, s=18, edgecolors="none")
ax.set_title(f"PCA of layer {LAYER} activations", fontsize=12)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# (3) Separation emerging across depth: PCA at several layers.
n_layers = store.shape[1]
probe_layers = np.linspace(0, n_layers - 1, 4, dtype=int)

fig, axes = plt.subplots(1, len(probe_layers), figsize=(4 * len(probe_layers), 4))
for ax, L in zip(axes, probe_layers):
    c2 = PCA(n_components=2).fit_transform(store[:, L, :].astype(np.float32))
    for lab, col in [(True, DECEPTIVE_C), (False, HONEST_C)]:
        m = labels == lab
        ax.scatter(c2[m, 0], c2[m, 1], c=col, alpha=0.5, s=10, edgecolors="none")
    ax.set_title(f"layer {L}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("PCA by layer  (red = deceptive, green = honest)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

Often the classes are tangled in early layers and pull apart in the middle-to-late layers, where the model has built up a higher-level representation of what it's saying. That's a hint about *where* to attach a probe — but PCA only shows two directions, so let's measure every layer directly.

### 2.2 Fit a probe per layer

We fit one logistic-regression probe per layer and score each with **cross-validated accuracy**, then pick the best layer and refit a deployable probe there.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from threadpoolctl import threadpool_limits   # ships with scikit-learn

def make_probe():
    return make_pipeline(
        StandardScaler(),
        PCA(n_components=256, random_state=0),
        LogisticRegression(max_iter=200, C=0.1),
    )

cv_acc = []
with threadpool_limits(limits=4):
    for L in range(n_layers):
        XL = store[:, L, :].astype(np.float32)
        cv_acc.append(cross_val_score(make_probe(), XL, labels, cv=5, scoring="accuracy", n_jobs=1).mean())
        print(f"  layer {L+1}/{n_layers}", end="\r")
print()

best_layer = int(np.argmax(cv_acc))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cv_acc, color="#3498db", lw=2, marker="o", ms=4)
ax.axvline(best_layer, color=DECEPTIVE_C, ls="--", lw=1.5,
           label=f"best layer {best_layer} ({cv_acc[best_layer]:.3f})")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.set_xlabel("layer"); ax.set_ylabel("probe accuracy (5-fold CV)")
ax.set_title("In-distribution linear probe accuracy by layer")
ax.set_ylim(0.4, 1.02); ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

# Fit the deployable probe on all data at the best layer.
probe = make_probe().fit(store[:, best_layer, :].astype(np.float32), labels)
print(f"Best in-distribution layer: {best_layer}  (CV accuracy {cv_acc[best_layer]:.3f})")

### 2.3 Make a prediction

You now have a trained probe. Using it is one call: featurize a prompt + response at `best_layer`, then ask the probe for its probability of deception.

In [ ]:
# Predict on the first few records (using their stored activations at best_layer).
X_best = store[:, best_layer, :].astype(np.float32)
scores = probe.predict_proba(X_best)[:, 1]   # P(deceptive)

for i in range(400,420):
    print(f"record {i}: P(deceptive) = {scores[i]:.2f}   (true label: {'deceptive' if labels[i] else 'honest'})")

## 3️⃣ Reading the results: AUROC and generalization

### 3.1 What is AUROC?

Accuracy hides two problems for a detector: it depends on an arbitrary threshold, and it flatters you when the classes are imbalanced. The standard metric instead is **AUROC** (Area Under the ROC Curve), and it has a clean interpretation:

> **AUROC = the probability that a randomly chosen *deceptive* example gets a higher score than a randomly chosen *honest* one.**

So:

- **1.0** — perfect: every lie scores above every honest response.
- **0.5** — chance: the probe's scores tell you nothing.
- **below 0.5** — *inverted*: the probe has found real signal but points the **wrong way** (it scores honest higher than deceptive). This is common when a probe transfers to a new scenario, and it's recoverable — but only if you notice it.

AUROC is **threshold-free** and **insensitive to class balance**, which is why we report it. Let's make the definition concrete on toy scores.

In [ ]:
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
honest_scores    = rng.normal(0.0, 1.0, 500)   # imaginary probe scores on honest examples
deceptive_scores = rng.normal(1.0, 1.0, 500)   # ... on deceptive ones (shifted higher)

s = np.concatenate([honest_scores, deceptive_scores])
y = np.concatenate([np.zeros(500), np.ones(500)])
print(f"AUROC = {roc_auc_score(y, s):.3f}")
print("Interpretation: a random deceptive example outscores a random honest one this fraction of the time.")

### 3.2 The idea that decides the competition: generalization

That ~0.9+ in-distribution accuracy from Section 2 is **not** your competition score, and chasing it will mislead you. Here's the trap:

- In-distribution accuracy is **near-saturated** — most layers score well, because within one scenario the probe can latch onto features specific to *that* scenario's prompts (templates, topic, phrasing) rather than deception itself.
- The competition tests your probe on **held-out lies it never trained on** — different scenarios, and ultimately the red-team **imposter models**. A probe that aces one scenario routinely **collapses to near-chance, or inverts below 0.5,** on another.

**Probe generalizability** is exactly this: does a probe trained on one distribution of lies still detect deception in a *different* one? That transfer number — not the in-distribution one — is what reflects competition difficulty.

And one discipline makes the transfer number trustworthy: **never select your layer (or any hyperparameter) on the data you report.** Use a held-out *validation* set to pick the layer, and a separate *test* set for the honest number.

Let's measure it directly: train on this scenario, then evaluate on a *different* one — splitting that target into a validation half (to pick the layer) and a test half (to report).

In [ ]:
from sklearn.model_selection import train_test_split

# Pick a DIFFERENT scenario (same captured model) as the transfer target.
scenarios = [p.split("scenario=")[-1].rstrip("/") for p in list_prefixes(f"data/captured_model={MODEL}/")]
TRANSFER_SCENARIO = next(s for s in scenarios if s != SCENARIO)
print(f"train on:    {SCENARIO}")
print(f"transfer to: {TRANSFER_SCENARIO}\n")

tbase   = f"captured_model={MODEL}/scenario={TRANSFER_SCENARIO}/split={SPLIT}"
df_t    = pd.read_parquet(f"s3://{BUCKET}/metadata/{tbase}/records.parquet", filesystem=fs)
y_t     = df_t["deceptive"].values
store_t = zarr.open(f"s3://{BUCKET}/data/{tbase}/activations.zarr", mode="r", storage_options={"anon": True})

# Split the transfer scenario: VAL half picks the layer, TEST half is the honest report.
val_idx, test_idx = train_test_split(np.arange(len(y_t)), test_size=0.5, stratify=y_t, random_state=0)

indist, val_auroc, test_auroc = [], [], []
for L in range(n_layers):
    clf = LogisticRegression(max_iter=1000, C=0.1).fit(store[:, L, :].astype(np.float32), labels)
    s_t = clf.decision_function(store_t[:, L, :].astype(np.float32))
    indist.append(roc_auc_score(labels, clf.decision_function(store[:, L, :].astype(np.float32))))
    val_auroc.append(roc_auc_score(y_t[val_idx],  s_t[val_idx]))
    test_auroc.append(roc_auc_score(y_t[test_idx], s_t[test_idx]))

sel_layer = int(np.argmax(val_auroc))   # chosen on validation ONLY
print(f"selected layer {sel_layer} on transfer-val (AUROC {val_auroc[sel_layer]:.3f})")
print(f"  -> honest transfer = test AUROC {test_auroc[sel_layer]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(indist,     color="#3498db",   lw=2, marker=".", label="in-distribution (train scenario)")
ax.plot(val_auroc,  color="#9b59b6",   lw=2, marker=".", label="transfer val (selection)")
ax.plot(test_auroc, color=DECEPTIVE_C, lw=2, marker=".", label="transfer test (reported)")
ax.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
ax.axvline(sel_layer, color="red", ls=":", lw=1, label=f"selected layer {sel_layer}")
ax.set_xlabel("layer"); ax.set_ylabel("AUROC"); ax.set_ylim(0, 1.02)
ax.set_title("In-distribution vs. transfer: the gap is the real story")
ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print(f"best in-distribution AUROC: {max(indist):.3f}  (optimistic — same data)")
print(f"reported transfer AUROC:    {test_auroc[sel_layer]:.3f}  <- reflects competition difficulty")

Read this plot the way a judge would:

- The **in-distribution** curve (blue) sits high and flat — near-saturated, and not informative about real performance.
- The **transfer** curves are the truth. If validation and test track each other and the good layers cluster together, the signal is real. If the argmax jumps around and val disagrees with test, no single layer truly transfers — and the honest takeaway is that this probe *doesn't* generalize here, regardless of which layer you pick.
- Watch for curves dipping **below 0.5**: that's an inverted probe — real signal pointing the wrong way.

The gap between the blue curve and the red one is the whole challenge of the competition in a single picture.

## 🧠 Where to go from here

This notebook used the **pre-computed** activations as a fast, GPU-free path. To go further:

- **Extract your own activations.** The pre-computed set covers a fixed list of models and scenarios. For your own prompts — or the red-team imposter models — run the model on NDIF and capture activations yourself. See [`2-predicting.ipynb`](2-predicting.ipynb).
- **Train on more scenarios, select on validation, report on test.** The same disciplines apply here as in the main notebook.

### Resources

- [Liars' Bench paper](https://arxiv.org/abs/2511.16035) (Kretschmar et al., 2025)
- [`Cadenza-Labs/liars-bench`](https://huggingface.co/datasets/Cadenza-Labs/liars-bench) — raw prompt + response pairs
- [nnsight docs](https://nnsight.net) and [NDIF](https://ndif.us) — remote activation extraction

---
*Clear all cell outputs before committing or sharing this notebook.*